# Gradio Day!

Today we will build User Interfaces using the outrageously simple Gradio framework.

Prepare for joy!

Please note: your Gradio screens may appear in 'dark mode' or 'light mode' depending on your computer settings.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from groq import Groq

In [2]:
import gradio as gr # oh yeah!

In [3]:
# Load environment variables
load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set")

Groq API Key exists and begins gsk_uqLa


In [4]:
# Initialize Groq client
groq_client = Groq()

In [5]:
# Let's wrap a call to llama-3.1-8b-instant in a simple function

system_message = "You are a helpful assistant"

def message_groq(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    completion = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        temperature=1,
        max_completion_tokens=1024,
        top_p=1
    )
    response = completion.choices[0].message.content
    response = response.replace('\u202f', ' ')
    return response

In [12]:
# This can reveal the "training cut off", or the most recent date in the training data

message_groq("What is today's date?")

'Today’s date is **March 12, 2026**.'

## User Interface time!

In [13]:
# here's a simple function

def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [14]:
shout("hello")

Shout has been called with input hello


'HELLO'

In [15]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">NOTE: Using Gradio's Share tool</h2>
            <span style="color:#900;">I'm about to show you a really cool way to share your Gradio UI with others. This deploys your gradio app as a demo on gradio's website, and then allows gradio to call the 'shout' function. This uses an advanced technology known as 'HTTP tunneling' (like ngrok for people who know it) which isn't allowed by many Antivirus programs and corporate environments. If you get an error, just skip the next cell.<br/>
            </span>
        </td>
    </tr>
</table>

In [16]:
# Adding share=True means that it can be accessed publically
# A more permanent hosting is available using a platform called Spaces from HuggingFace, which we will touch on next week
# NOTE: Some Anti-virus software and Corporate Firewalls might not like you using share=True. 
# If you're at work on on a work network, I suggest skip this test.

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://1c62c78fb07e860afd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Adding inbrowser=True opens up a new browser window automatically

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


gio: http://127.0.0.1:7862/: Operation not supported


## Adding authentication

Gradio makes it very easy to have userids and passwords

Obviously if you use this, have it look properly in a secure place for passwords! At a minimum, use your .env

In [ ]:
# Adding authentication

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(inbrowser=True, auth=("jay", "bananas"))

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


gio: http://127.0.0.1:7863/: Operation not supported


## Forcing dark mode

Gradio appears in light mode or dark mode depending on the settings of the browser and computer. There is a way to force gradio to appear in dark mode, but Gradio recommends against this as it should be a user preference (particularly for accessibility reasons). But if you wish to force dark mode for your screens, below is how to do it.

In [19]:
# Forcing dark mode

force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [20]:
# Adding a little more:

message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [24]:
# And now - changing the function from "shout" to "message_groq"

message_input = gr.Textbox(label="Your message:", info="Enter a message for gpt-oss-120b", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=message_groq,
    title="Groq LLM", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [23]:
# Let's use Markdown
# Using global system_message variable

system_message = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your message:", info="Enter a message for gpt-oss-120b", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_groq,
    title="Groq LLM", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


In [25]:
# Let's create a call that streams back results

def stream_groq(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = groq_client.chat.completions.create(
        model='openai/gpt-oss-120b',
        messages=messages,
        stream=True,
        temperature=1,
        max_completion_tokens=1024,
        top_p=1
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [26]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for gpt-oss-120b", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_groq,
    title="Groq LLM - Streaming", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


In [29]:
# Stream with different Groq models

def stream_model(prompt, model_name, reasoning_level):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    
    model_map = {
        "Llama 3.1 8B": "llama-3.1-8b-instant",
        "Llama 3.3 70B": "llama-3.3-70b-versatile",
        "Llama 4 Scout": "meta-llama/llama-4-scout-17b-16e-instruct",
        "Kimi K2": "moonshotai/kimi-k2-instruct",
        "GPT-OSS 20B": "openai/gpt-oss-20b",
        "GPT-OSS 120B": "openai/gpt-oss-120b",
        "Qwen 32B": "qwen/qwen3-32b"
    }
    
    model = model_map.get(model_name, "llama-3.1-8b-instant")
    
    # Base kwargs
    kwargs = {
        "model": model,
        "messages": messages,
        "stream": True,
        "temperature": 1,
        "max_completion_tokens": 6000,
        "top_p": 1
    }
    
    # Add reasoning_effort based on model type
    if "gpt-oss" in model:
        # GPT supports low, medium, high
        kwargs["reasoning_effort"] = reasoning_level.lower()
    elif "qwen" in model:
        # Qwen supports default or none
        if reasoning_level == "None":
            kwargs["reasoning_effort"] = "none"
        else:
            kwargs["reasoning_effort"] = "default"
    
    stream = groq_client.chat.completions.create(**kwargs)
    
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [30]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(
    ["Llama 3.1 8B", "Llama 3.3 70B", "Llama 4 Scout", "Kimi K2", "GPT-OSS 20B", "GPT-OSS 120B", "Qwen 32B"], 
    label="Select model", 
    value="Llama 3.1 8B"
)

# Reasoning level dropdown - will be contextually applied based on model
reasoning_selector = gr.Dropdown(
    ["None", "Low", "Medium", "High"], 
    label="Reasoning Effort (GPT-OSS: low/medium/high, Qwen: default/none)", 
    value="Medium"
)

message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="Groq LLM Playground", 
    inputs=[message_input, model_selector, reasoning_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "Llama 3.1 8B", "Medium"],
            ["Solve this math problem: 27 * 14 + 35", "GPT-OSS 120B", "High"],
            ["Write a haiku about artificial intelligence", "Qwen 32B", "Default"]
        ], 
    flagging_mode="never"
    )
view.launch()

/home/jay/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: Default or set allow_custom_value=True.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


# Building a company brochure generator

In [7]:
from scraper import fetch_website_contents

In [8]:
# Update system message for brochure generation

system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [9]:
def stream_brochure(company_name, url, model_name, reasoning_level):
    yield "Fetching website content...\n"
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    
    model_map = {
        "Llama 3.1 8B": "llama-3.1-8b-instant",
        "Llama 3.3 70B": "llama-3.3-70b-versatile",
        "Llama 4 Scout": "meta-llama/llama-4-scout-17b-16e-instruct",
        "Kimi K2": "moonshotai/kimi-k2-instruct",
        "GPT-OSS 20B": "openai/gpt-oss-20b",
        "GPT-OSS 120B": "openai/gpt-oss-120b",
        "Qwen 32B": "qwen/qwen3-32b"
    }
    
    model = model_map.get(model_name, "llama-3.1-8b-instant")
    
    kwargs = {
        "model": model,
        "messages": messages,
        "stream": True,
        "temperature": 0.7,
        "max_completion_tokens": 2048,
        "top_p": 1
    }
    
    # Add reasoning_effort based on model type
    if "gpt-oss" in model:
        # GPT supports low, medium, high
        kwargs["reasoning_effort"] = reasoning_level.lower()
    elif "qwen" in model:
        # Qwen supports default or none
        if reasoning_level == "None":
            kwargs["reasoning_effort"] = "none"
        else:
            kwargs["reasoning_effort"] = "default"
    
    stream = groq_client.chat.completions.create(**kwargs)
    
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(
    ["Llama 3.1 8B", "Llama 3.3 70B", "Llama 4 Scout", "Kimi K2", "GPT-OSS 20B", "GPT-OSS 120B", "Qwen 32B"], 
    label="Select model", 
    value="Llama 3.1 8B"
)

# Add reasoning selector for brochure too
reasoning_selector = gr.Dropdown(
    ["None", "Low", "Medium", "High"], 
    label="Reasoning Effort (GPT-OSS: low/medium/high, Qwen: default/none)", 
    value="Medium"
)

message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector, reasoning_selector], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT-OSS 120B", "High"],
            ["OpenAI", "https://openai.com", "Llama 3.3 70B", "Medium"],
            ["Anthropic", "https://anthropic.com", "Qwen 32B", "Default"]
        ], 
    flagging_mode="never"
    )
view.launch()

/home/jay/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: Default or set allow_custom_value=True.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/jay/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 849, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jay/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jay/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jay/llm_engineering/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1710, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jay/llm_engineering/llm_engi